### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 7강 실습 — Autonomy Level 설계 & HITL Escalation

> 목표: 에이전트에 **자율성 레벨(0~4)** 개념을 적용하고, **3가지 에스컬레이션 트리거(신뢰도·비용·위험)**와
> **Approval Queue(사람 승인)**를 구현해 **Level 2 Sweet Spot**(대부분 자동, 위험할 때만 사람)을 만듭니다.
>
> 산출물: 트리거 정책 · Approval Queue · L2 자율 에이전트(환불 시나리오)

## 0. 셋업

In [6]:
# ── 셋업 ──
!pip -q install openai

In [7]:
# ── 셋업 ──
# === Upstage Solar 셋업 ===

from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('UPSTAGE_API_KEY')

client = OpenAI(api_key=api_key, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro3"
print("Solar 준비 완료:", MODEL)

Solar 준비 완료: solar-pro3


## 1. 자율성 레벨 0~4
| Level | 설명 | 비용 | 위험 | 속도 |
|---|---|---|---|---|
| 0 Manual | 모든 행동 사람 승인 | 높음 | 최저 | 느림 |
| 1 Draft | 에이전트 초안, 단계마다 승인 | 중상 | 낮음 | 느림 |
| **2 Supervised ★** | 자율 실행 + 위험할 때만 에스컬레이션 | 중 | 중 | 빠름 |
| 3 Oversight | 전면 자율, 사람은 모니터만 | 낮음 | 높음 | 빠름 |
| 4 Full | 완전 자율 | 최저 | 최고 | 최빠름 |

**Level 2~3가 Production Sweet Spot** — 신뢰성 기술이 받치지 못하는 상황입니다.

## 2. 3가지 에스컬레이션 트리거
'언제 사람을 부를까'가 핵심. 신뢰도·비용·위험 셋 중 하나라도 걸리면 사람에게 알려지도록(에스컬레이션) 세팅합니다.

In [9]:
# ── 3가지 에스컬레이션 트리거 ──
RISKY = {"refund", "send_email", "delete", "transfer"}  # 되돌리기 어려운 작업
COST_LIMIT = 100_000   # 원
CONF_MIN = 0.70

def should_escalate(action, amount=0, confidence=1.0):
    """위험작업이면 항상, 고액이면, 신뢰도 낮으면 → 사람에게 에스컬레이션."""
    reasons=[]
    if action in RISKY:        reasons.append(f"위험작업({action})")
    if amount > COST_LIMIT:    reasons.append(f"고액({amount:,}원>{COST_LIMIT:,})")
    if confidence < CONF_MIN:  reasons.append(f"낮은 신뢰도({confidence:.2f})")
    return (len(reasons)>0, reasons)

print(should_escalate("search", confidence=0.9))            # 자동(안전)
print(should_escalate("refund", amount=30_000))             # 위험작업 → 승인
print(should_escalate("refund", amount=300_000))            # 위험+고액 → 승인

(False, [])
(True, ['위험작업(refund)'])
(True, ['위험작업(refund)', '고액(300,000원>100,000)'])


## 3. Approval Queue (HITL) + Level 2 에이전트
에스컬레이션되면 **대기열**에 넣고 사람이 승인 혹은 거부합니다. 안전 작업은 자동으로 실행됩니다.

In [4]:
# ── Approval Queue (HITL) + Level 2 에이전트 ──
class ApprovalQueue:
    def __init__(self): self.pending=[]
    def submit(self, action, payload, reasons):
        ticket={"id":len(self.pending)+1,"action":action,"payload":payload,"reasons":reasons,"status":"PENDING"}
        self.pending.append(ticket); return ticket
    def resolve(self, tid, approve):
        t=self.pending[tid-1]; t["status"]="APPROVED" if approve else "REJECTED"; return t

queue=ApprovalQueue()
def execute(action, payload): return f"✅ 실행됨: {action}({payload})"

def l2_agent(action, payload, amount=0, confidence=1.0):
    esc, reasons = should_escalate(action, amount, confidence)
    if not esc:
        return execute(action, payload)
    t=queue.submit(action, payload, reasons)
    return f"⏸ 에스컬레이션 → 승인대기 #{t['id']} (사유: {', '.join(reasons)})"

print(l2_agent("search", {"q":"주문조회"}))                  # 자동(안전)
print(l2_agent("refund", {"order":"A-12"}, amount=30_000))   # 위험 → 승인대기 #1
print(l2_agent("refund", {"order":"B-99"}, amount=500_000))  # 위험+고액 → 승인대기 #2

✅ 실행됨: search({'q': '주문조회'})
⏸ 에스컬레이션 → 승인대기 #1 (사유: 위험작업(refund))
⏸ 에스컬레이션 → 승인대기 #2 (사유: 위험작업(refund), 고액(500,000원>100,000))


## 4. 사람이 승인 → 실행

In [10]:
# ── 사람이 승인 → 실행 ──
print("대기열:", [(t["id"],t["action"],t["status"]) for t in queue.pending])
t=queue.resolve(1, approve=True)
print(f"#{t['id']} {t['status']} →", execute(t["action"],t["payload"]) if t["status"]=="APPROVED" else "취소")
t2=queue.resolve(2, approve=False)
print(f"#{t2['id']} {t2['status']} → 실행 안 함(고액 환불 거부)")

대기열: [(1, 'refund', 'APPROVED'), (2, 'refund', 'REJECTED')]
#1 APPROVED → ✅ 실행됨: refund({'order': 'A-12'})
#2 REJECTED → 실행 안 함(고액 환불 거부)


## 정리
- **Autonomy Level 2**(자율+위험시 승인)가 프로덕션 Sweet Spot.
- **신뢰도·비용·위험** 트리거로 '언제 사람을 부를지' 결정.
- **Approval Queue**로 위험 작업을 사람 손에 남긴다(되돌릴 수 없는 일).

**NEXT:** 이 자율 에이전트가 **실패에도 안 무너지도록** Reliability Layer를 씌워 봅니다.